In [1]:
import mne
import matplotlib.pyplot as plt

raw: mne.io.Raw = mne.io.read_raw_edf("data/eeg_test.edf")
# print(raw)
# print(raw.info)
# print(raw.ch_names)
eeg_data = raw.get_data()
print(eeg_data.shape)
raw.plot(n_channels = len(raw.ch_names), duration=20.0, start=10.0) # отобразить 20 секунд записи начиная с десятой секунды

Extracting EDF parameters from /Users/paks/Polytech/eeg_practice/ArtifactRemovalTransformer/data/eeg_test.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
(22, 495500)
Using qt as 2D backend.
Using pyopengl with version 3.1.9


In [2]:
print(raw.ch_names)

['EEG FP1-A1', 'EEG FP2-A2', 'EEG F3-A1', 'EEG F4-A2', 'EEG C3-A1', 'EEG C4-A2', 'EEG P3-A1', 'EEG P4-A2', 'EEG O1-A1', 'EEG O2-A2', 'EEG F7-A1', 'EEG F8-A2', 'EEG T3-A1', 'EEG T4-A2', 'EEG T5-A1', 'EEG T6-A2', 'EEG FPZ-A1', 'EEG FZ-A2', 'EEG CZ-A1', 'EEG PZ-A2', 'EEG OZ-A1', 'ECG  ECG']


In [3]:
channel_names = raw.ch_names
ch_names_cleaned = [ch.split()[1].split('-')[0] if '-' in ch else ch.split()[-1] for ch in channel_names]

print(ch_names_cleaned)

['FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FPZ', 'FZ', 'CZ', 'PZ', 'OZ', 'ECG']


In [4]:
import pandas as pd
from typing import Final



loc_df = pd.read_csv("template_chanlocs.loc", sep='\t', header=None)
channel_names_from_chanlocs = loc_df.iloc[:, -1].tolist()
names_stripped_from_locs = [name.replace(' ', '').upper() for name in channel_names_from_chanlocs]
names_stripped_from_locs

FileNotFoundError: [Errno 2] No such file or directory: 'template_chanlocs.loc'

Channels marked as bad:
none


In [100]:
ch_names_cleaned_set = set(ch_names_cleaned)
names_stripped_from_locs_set = set(names_stripped_from_locs)

only_in_locs = names_stripped_from_locs_set - ch_names_cleaned_set
only_in_cleaned = ch_names_cleaned_set - names_stripped_from_locs_set

common = ch_names_cleaned_set & names_stripped_from_locs_set

print("Only in locs: ", only_in_locs, ", len of locs = ", len(names_stripped_from_locs_set))
print("Only in cleaned: ", only_in_cleaned, ", len of cleaned = ", len(ch_names_cleaned_set))
print("Common: ", common)


Only in locs:  {'FT7', 'TP8', 'TP7', 'CP4', 'CPZ', 'CP3', 'FCZ', 'FT8', 'FC3', 'FC4'} , len of locs =  30
Only in cleaned:  {'ECG', 'FPZ'} , len of cleaned =  22
Common:  {'P3', 'FP1', 'T5', 'FP2', 'O1', 'T4', 'F7', 'T6', 'F4', 'CZ', 'C4', 'O2', 'OZ', 'PZ', 'C3', 'FZ', 'P4', 'F8', 'F3', 'T3'}


In [34]:
import numpy as np

In [37]:
np.array(eeg_data.data).shape

(22, 495500)

In [101]:
channel_names = ch_names_cleaned

In [106]:
channels_to_remove = {'ECG'}

In [107]:
keep_indices = [i for i, name in enumerate(channel_names) if name.upper() not in channels_to_remove]
filtered_data = eeg_data[keep_indices, :]
our_channels_names = [channel_names[i] for i in keep_indices]
our_channels_names

['FP1',
 'FP2',
 'F3',
 'F4',
 'C3',
 'C4',
 'P3',
 'P4',
 'O1',
 'O2',
 'F7',
 'F8',
 'T3',
 'T4',
 'T5',
 'T6',
 'FPZ',
 'FZ',
 'CZ',
 'PZ',
 'OZ']

In [108]:
subset_data = filtered_data[:, :1000]

In [109]:
df = pd.DataFrame(subset_data.T, columns=our_channels_names)

df.T.to_csv("eeg_subset.csv", index=False, header=False)

In [96]:
# Assuming you already have ch_names_cleaned and names_stripped_from_locs lists

# Initialize the dictionary to store matches
index_map = {}

# Iterate over the names_stripped_from_locs list
for i, name in enumerate(names_stripped_from_locs):
    if name in ch_names_cleaned:
        # Get the index of the matching name in ch_names_cleaned
        j = ch_names_cleaned.index(name)
        # Add the pair (index from names_stripped_from_locs, index from ch_names_cleaned) to the dictionary
        index_map[i] = j

# The dictionary index_map now contains the matching indices
print(index_map.values())


dict_values([0, 1, 10, 2, 17, 3, 11, 12, 4, 18, 5, 13, 14, 6, 19, 7, 15, 8, 20, 9])


In [91]:
filtered_loc_df.index.map(index_map)

Index([0, 1, 10, 2, 17, 3, 11, 12, 4, 18, 5, 13, 14, 6, 19, 7, 15, 8, 20, 9], dtype='int64')